In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import matplotlib.pyplot as plt
import imageio
from google.colab import files
from google.colab import drive
import os

class Env:
    def __init__(self, size=10):
        self.size = size
        self.walls = {(3,3),(3,4),(4,4),(5,4),(6,4)}
        self.reset()

    def reset(self):
        self.Caine = (0,0)
        self.Abel = (9,9)
        return self.state()

    def state(self):
        grid = np.zeros((3,self.size,self.size))

        cx, cy = self.Caine
        ax, ay = self.Abel

        grid[0,cy,cx] = 1
        grid[1,ay,ax] = 1

        for x,y in self.walls:
            grid[2,y,x] = 1

        return grid.flatten()

    def move(self, pos, a):
        x, y = pos
        dx, dy = [(0,-1),(0,1),(-1,0),(1,0)][a]
        nx, ny = x+dx, y+dy

        nx = max(0,min(self.size-1,nx))
        ny = max(0,min(self.size-1,ny))

        if (nx,ny) in self.walls:
            return (x,y)

        return (nx,ny)

    def step(self, aCaine, aAbel):
        self.Caine = self.move(self.Caine, aCaine)
        self.Abel = self.move(self.Abel, aAbel)

        done = False
        rCaine = -0.01
        rAbel = 0.01

        if self.Caine == self.Abel:
            rCaine = 10
            rAbel = -10
            done = True

        return self.state(), rCaine, rAbel, done


class DQN(nn.Module):
    def __init__(self, inp):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inp,128),
            nn.ReLU(),
            nn.Linear(128,128),
            nn.ReLU(),
            nn.Linear(128,4)
        )

    def forward(self, x):
        return self.net(x)


def render(env):
    img = np.ones((env.size,env.size,3))

    for x,y in env.walls:
        img[y,x] = [0,0,0]

    cx, cy = env.Caine
    ax, ay = env.Abel

    img[cy,cx] = [1,0,0]
    img[ay,ax] = [0,0,1]

    return img


def train_and_record():
    env = Env()

    Caine = DQN(env.state().shape[0])
    Abel = DQN(env.state().shape[0])

    optCaine = optim.Adam(Caine.parameters(), 1e-3)
    optAbel = optim.Adam(Abel.parameters(), 1e-3)

    frames = []
    epsilon = 1.0

    for ep in range(300):
        state = env.reset()
        done = False

        while not done:
            def act(model):
                if random.random() < epsilon:
                    return random.randint(0,3)
                with torch.no_grad():
                    return int(torch.argmax(model(torch.tensor(state,dtype=torch.float32))))

            aCaine = act(Caine)
            aAbel = act(Abel)

            state, rCaine, rAbel, done = env.step(aCaine, aAbel)

            qCaine = Caine(torch.tensor(state,dtype=torch.float32))[aCaine]
            lossCaine = (qCaine - rCaine)**2
            optCaine.zero_grad()
            lossCaine.backward()
            optCaine.step()

            qAbel = Abel(torch.tensor(state,dtype=torch.float32))[aAbel]
            lossAbel = (qAbel - rAbel)**2
            optAbel.zero_grad()
            lossAbel.backward()
            optAbel.step()

            frames.append(render(env))

        epsilon *= 0.995

        if ep % 20 == 0:
            print("episode:", ep)

    print("saving video")

    imageio.mimsave(
        "rl_duel.mp4",
        (np.array(frames) * 255).astype(np.uint8),
        fps=15
    )

    print("DONE → rl_duel.mp4")

    return Caine, Abel, frames


Caine, Abel, frames = train_and_record()

drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/rl_circus/"
os.makedirs(DRIVE_PATH, exist_ok=True)

torch.save({
    "model_Caine": Caine.state_dict(),
    "model_Abel": Abel.state_dict(),
}, DRIVE_PATH + "best_agents.pth")

video_path = DRIVE_PATH + "rl_duel.mp4"

imageio.mimsave(
    video_path,
    (np.array(frames) * 255).astype("uint8"),
    fps=15
)

files.download(video_path)

print("🎪 Сохранено в:")
print("Model:", DRIVE_PATH + "best_agents.pth")
print("Video:", video_path)